<a href="https://colab.research.google.com/github/ananda-ayu-kartika23/Streamlit/blob/main/Streamlit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
!pip install streamlit pyngrok --quiet

In [15]:
%%writefile apps.py
import streamlit as st
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

@st.cache_data
def load_data():
    df = pd.read_csv("Mall_Customers.csv")
    return df

@st.cache_data
def train_model(df, n_clusters):
    # Fitur yang dipakai untuk clustering
    features = df[["Gender", "Age", "Annual Income (k$)", "Spending Score (1-100)"]]

    # Preprocessing: Gender di-encode, numerik di-scaling
    preprocess = ColumnTransformer(
        transformers=[
            ("cat", OneHotEncoder(drop="first"), ["Gender"]),
            ("num", StandardScaler(), ["Age", "Annual Income (k$)", "Spending Score (1-100)"])
        ]
    )

    model = Pipeline(steps=[
        ("preprocess", preprocess),
        ("cluster", KMeans(n_clusters=n_clusters, random_state=42, n_init=10))
    ])

    model.fit(features)
    return model

def main():
    st.set_page_config(page_title="Clustering Customer Mall", layout="centered")
    st.title("🛍️ Aplikasi Clustering Mall Customers")
    st.write("Masukkan data customer untuk melihat cluster-nya.")

    df = load_data()

    st.sidebar.header("Pengaturan Cluster")
    n_clusters = st.sidebar.slider("Jumlah Cluster", 2, 8, 4)

    model = train_model(df, n_clusters)

    gender = st.selectbox("Gender", ["Male", "Female"])
    age = st.slider("Age", int(df["Age"].min()), int(df["Age"].max()), int(df["Age"].mean()))
    annual_income = st.slider(
        "Annual Income (k$)",
        int(df["Annual Income (k$)"].min()),
        int(df["Annual Income (k$)"].max()),
        int(df["Annual Income (k$)"].mean())
    )
    spending_score = st.slider(
        "Spending Score (1-100)",
        int(df["Spending Score (1-100)"].min()),
        int(df["Spending Score (1-100)"].max()),
        int(df["Spending Score (1-100)"].mean())
    )

    if st.button("Prediksi Cluster"):
        input_data = pd.DataFrame({
            "Gender": [gender],
            "Age": [age],
            "Annual Income (k$)": [annual_income],
            "Spending Score (1-100)": [spending_score]
        })

        cluster = model.predict(input_data)[0]
        st.success(f"Customer ini masuk ke **Cluster {cluster}**")

    if st.checkbox("Tampilkan Dataset"):
        st.dataframe(df)

if __name__ == "__main__":
    main()

Overwriting apps.py


In [16]:
from pyngrok import ngrok
ngrok.set_auth_token("3DF0dAPZTTGWH7Zei6sYHm3oEhr_3rki5TxagATji1JPqTuGV")
public_url = ngrok.connect(8501)
print(f"Streamlit app bisa diakses di:\n{public_url}")

!streamlit run apps.py &>/dev/null&

Streamlit app bisa diakses di:
NgrokTunnel: "https://magnify-reconcile-pediatric.ngrok-free.dev" -> "http://localhost:8501"
